# Incident Response Runbook: Exfiltration - Exfiltration Over Alternative Protocol

**Tactic:** Exfiltration
**Technique:** Exfiltration Over Alternative Protocol (T1048)
**Severity:** HIGH

## Overview

This runbook provides step-by-step incident response procedures for Exfiltration Over Alternative Protocol activities.

## Incident Response Phases

1. **Detection & Analysis**
2. **Containment**
3. **Eradication**
4. **Recovery**
5. **Post-Incident Activities**


## Phase 1: Detection & Analysis

### Objectives
- Confirm the incident
- Identify the scope
- Assess impact


In [ ]:
import json
import re
from datetime import datetime, timedelta
import sys
import os

# Add parent directory to path for imports
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

from splunk.splunk_data_collector import SplunkDataCollector
from crowdstrike.crowdstrike_data_collector import CrowdStrikeDataCollector
from misp.misp_integration import MISPIntegration
from iris.iris_integration import IRISIntegration
from shuffle.shuffle_integration import ShuffleIntegration

# Step 1: Initial Alert Analysis & Detection
print("=" * 60)
print("STEP 1: Initial Alert Analysis & Detection")
print("=" * 60)

# Initialize security integrations
splunk = SplunkDataCollector()
crowdstrike = CrowdStrikeDataCollector()
misp = MISPIntegration()
iris = IRISIntegration()
shuffle = ShuffleIntegration()

# Create incident case in IRIS
incident_data = {
    'title': 'Data Exfiltration Over Alternative Protocol - T1048',
    'description': 'Potential data exfiltration using alternative protocols detected',
    'severity': 'HIGH',
    'category': 'Data Exfiltration',
    'tags': ['exfiltration', 'alternative-protocol', 'data-theft', 't1048']
}

case_id = iris.create_case(incident_data)
print(f" Created IRIS case: {case_id}")

# Detection queries for alternative protocol exfiltration
print("\n Executing alternative protocol exfiltration detection queries...")

# Splunk query for unusual protocol usage
protocol_anomaly_query = """
index=* sourcetype=network_traffic
| where (protocol="ICMP" AND bytes > 1000) OR (protocol="DNS" AND bytes > 2000)
OR (dest_port IN (53, 80, 443) AND bytes > 1000000)
OR (protocol IN ("NTP", "SNMP", "Syslog") AND bytes > 50000)
| stats sum(bytes) as total_bytes, count by src_ip, dest_ip, protocol, dest_port
| where total_bytes > 1000000
| sort -total_bytes
"""

protocol_results = splunk.execute_query(protocol_anomaly_query)
print(f" Found {len(protocol_results)} protocol anomalies")

# Splunk query for DNS tunneling
dns_tunnel_query = """
index=* sourcetype=dns_logs
| where query_length > 200 OR subdomain_count > 10
| eval domain_parts = split(query, ".")
| eval subdomain_count = mvcount(domain_parts) - 2
| where subdomain_count > 5
| stats count, sum(query_length) as total_length by src_ip, query
| where count > 10 OR total_length > 50000
"""

dns_results = splunk.execute_query(dns_tunnel_query)
print(f" Found {len(dns_results)} potential DNS tunneling activities")

# Splunk query for ICMP exfiltration
icmp_exfil_query = """
index=* sourcetype=network_traffic
| where protocol="ICMP"
| eval packet_size = bytes
| where packet_size > 1000
| stats sum(bytes) as total_bytes, count by src_ip, dest_ip, packet_size
| where total_bytes > 500000
| sort -total_bytes
"""

icmp_results = splunk.execute_query(icmp_exfil_query)
print(f" Found {len(icmp_results)} ICMP exfiltration patterns")

# Splunk query for large transfers over unusual ports
unusual_port_query = """
index=* sourcetype=network_traffic
| where NOT dest_port IN (80, 443, 53, 25, 110, 143, 993, 995, 22, 21, 20, 23, 3389)
| where bytes > 500000
| stats sum(bytes) as total_bytes, count by src_ip, dest_ip, dest_port, protocol
| where total_bytes > 1000000
| sort -total_bytes
"""

port_results = splunk.execute_query(unusual_port_query)
print(f" Found {len(port_results)} large transfers over unusual ports")

# CrowdStrike query for suspicious outbound connections
cs_exfil_query = """
event_simpleName=NetworkConnectIP4 OR event_simpleName=NetworkConnectIP6
| where ConnectionDirection = "Outbound"
| where (Protocol="ICMP" AND Bytes > 1000) OR (Protocol="UDP" AND Port != 53 AND Bytes > 100000)
| stats sum(Bytes) as total_bytes by ComputerName, RemoteAddress, Protocol, Port
| where total_bytes > 500000
"""

cs_exfil_results = crowdstrike.execute_query(cs_exfil_query)
print(f" Found {len(cs_exfil_results)} suspicious outbound connections")

# Check for known exfiltration IOCs in MISP
exfil_indicators = misp.search_indicators("exfiltration", limit=50)
print(f" Retrieved {len(exfil_indicators)} exfiltration indicators from MISP")

# Compile detection results
detection_summary = {
    'case_id': case_id,
    'timestamp': datetime.now().isoformat(),
    'tactic': 'Exfiltration',
    'technique': 'Exfiltration Over Alternative Protocol (T1048)',
    'severity': 'HIGH',
    'indicators': {
        'protocol_anomalies': len(protocol_results),
        'dns_tunneling': len(dns_results),
        'icmp_exfiltration': len(icmp_results),
        'unusual_port_transfers': len(port_results),
        'suspicious_connections': len(cs_exfil_results),
        'misp_indicators': len(exfil_indicators)
    },
    'affected_hosts': list(set([r.get('src_ip', r.get('ComputerName', 'unknown')) 
                               for r in protocol_results + dns_results + icmp_results + port_results + cs_exfil_results])),
    'data_volume': sum([r.get('total_bytes', 0) for r in protocol_results + icmp_results + port_results])
}

print(f"\n📊 Detection Summary:")
print(f"  Case ID: {detection_summary['case_id']}")
print(f"  Severity: {detection_summary['severity']}")
print(f"  Affected Hosts: {len(detection_summary['affected_hosts'])}")
print(f"  Protocol Anomalies: {detection_summary['indicators']['protocol_anomalies']}")
print(f"  DNS Tunneling: {detection_summary['indicators']['dns_tunneling']}")
print(f"  ICMP Exfiltration: {detection_summary['indicators']['icmp_exfiltration']}")
print(f"  Unusual Port Transfers: {detection_summary['indicators']['unusual_port_transfers']}")
print(f"  Data Volume: {detection_summary['data_volume']:,} bytes")

# Trigger Shuffle workflow for automated exfiltration response
if detection_summary['indicators']['protocol_anomalies'] > 0 or detection_summary['data_volume'] > 1000000:
    shuffle.trigger_workflow('exfiltration_detection_response', detection_summary)
    print(" Triggered automated exfiltration response workflow")


## Phase 2: Containment

### Objectives
- Stop the attack
- Isolate affected systems
- Prevent spread


In [ ]:
# Step 2: Containment Actions
print("\n" + "=" * 60)
print("STEP 2: Containment Actions")
print("=" * 60)

containment_actions = []

# Block suspicious destination IPs
print("\n Blocking suspicious destination IPs...")
all_suspicious_ips = set()
for result in protocol_results + icmp_results + port_results:
    dest_ip = result.get('dest_ip')
    if dest_ip:
        all_suspicious_ips.add(dest_ip)

for ip in list(all_suspicious_ips)[:50]:  # Limit to top 50
    try:
        block_result = shuffle.block_ip(ip)
        if block_result.get('success'):
            containment_actions.append(f" Blocked destination IP: {ip}")
            print(f"   Blocked destination IP: {ip}")
    except Exception as e:
        containment_actions.append(f" Failed to block IP {ip}: {str(e)}")
        print(f"   Failed to block IP {ip}: {str(e)}")

# Isolate affected hosts via CrowdStrike
print("\n Isolating affected hosts...")
for host in detection_summary['affected_hosts']:
    try:
        isolation_result = crowdstrike.isolate_host(host)
        if isolation_result.get('success'):
            containment_actions.append(f" Isolated host: {host}")
            print(f"   Isolated host: {host}")
        else:
            containment_actions.append(f" Failed to isolate host: {host}")
            print(f"   Failed to isolate host: {host}")
    except Exception as e:
        containment_actions.append(f" Error isolating {host}: {str(e)}")
        print(f"   Error isolating {host}: {str(e)}")

# Block alternative protocol abuse
print("\n Blocking alternative protocol abuse...")
try:
    protocol_blocks = {
        'block_large_icmp': True,
        'block_dns_tunneling': True,
        'block_unusual_port_transfers': True,
        'rate_limit_suspicious_protocols': True
    }
    protocol_result = shuffle.block_protocol_abuse(protocol_blocks)
    if protocol_result.get('success'):
        containment_actions.append(" Alternative protocol abuse blocked")
        print("   Alternative protocol abuse blocked")
except Exception as e:
    containment_actions.append(f" Failed to block protocol abuse: {str(e)}")
    print(f"   Failed to block protocol abuse: {str(e)}")

# Enable data loss prevention (DLP)
print("\n Enabling data loss prevention...")
try:
    dlp_config = {
        'block_large_outbound_transfers': True,
        'monitor_suspicious_protocols': True,
        'alert_on_data_exfiltration': True,
        'quarantine_suspicious_files': True
    }
    dlp_result = shuffle.enable_dlp(dlp_config)
    if dlp_result.get('success'):
        containment_actions.append(" Data loss prevention enabled")
        print("   Data loss prevention enabled")
except Exception as e:
    containment_actions.append(f" Failed to enable DLP: {str(e)}")
    print(f"   Failed to enable DLP: {str(e)}")

# Implement traffic throttling
print("\n⏱ Implementing traffic throttling...")
try:
    throttle_config = {
        'throttle_icmp_traffic': True,
        'throttle_dns_queries': True,
        'throttle_unusual_ports': True,
        'bandwidth_limit': 1000000  # 1MB/s limit for suspicious traffic
    }
    throttle_result = shuffle.implement_traffic_throttling(throttle_config)
    if throttle_result.get('success'):
        containment_actions.append(" Traffic throttling implemented")
        print("   Traffic throttling implemented")
except Exception as e:
    containment_actions.append(f" Failed to implement traffic throttling: {str(e)}")
    print(f"   Failed to implement traffic throttling: {str(e)}")

# Update IRIS case with containment actions
iris.update_case(case_id, {
    'containment_actions': containment_actions,
    'containment_timestamp': datetime.now().isoformat(),
    'status': 'containment_in_progress'
})

print(f"\n Containment Summary:")
print(f"  Actions Taken: {len([a for a in containment_actions if a.startswith('')])}")
print(f"  Warnings: {len([a for a in containment_actions if a.startswith('')])}")
print(f"  Errors: {len([a for a in containment_actions if a.startswith('')])}")

# Trigger Shuffle workflow for containment verification
shuffle.trigger_workflow('exfiltration_containment_verification', {
    'case_id': case_id,
    'actions_taken': containment_actions
})
print(" Triggered containment verification workflow")


## Phase 3: Eradication

### Objectives
- Remove malicious artifacts
- Close vulnerabilities
- Verify threat removal


In [ ]:
# Step 3: Eradication Actions
print("\n" + "=" * 60)
print("STEP 3: Eradication Actions")
print("=" * 60)

eradication_actions = []

# Terminate exfiltration processes via CrowdStrike
print("\n💀 Terminating exfiltration processes...")
for connection in cs_exfil_results:
    try:
        # Find and terminate processes associated with suspicious connections
        process_query = f"""
        event_simpleName=NetworkConnectIP4
        | where RemoteAddress="{connection['RemoteAddress']}"
        | stats latest(ProcessId) by ComputerName, ProcessId
        """
        process_info = crowdstrike.execute_query(process_query)
        for proc in process_info:
            kill_result = crowdstrike.terminate_process(proc['ComputerName'], proc['ProcessId'])
            if kill_result.get('success'):
                eradication_actions.append(f" Terminated exfiltration process: PID {proc['ProcessId']} on {proc['ComputerName']}")
                print(f"   Terminated exfiltration process: PID {proc['ProcessId']} on {proc['ComputerName']}")
    except Exception as e:
        eradication_actions.append(f" Failed to terminate processes for {connection['RemoteAddress']}: {str(e)}")
        print(f"   Failed to terminate processes for {connection['RemoteAddress']}: {str(e)}")

# Remove exfiltration tools and malware
print("\n🗑 Removing exfiltration tools...")
for host in detection_summary['affected_hosts']:
    try:
        # Look for common exfiltration tools
        tool_scan = crowdstrike.scan_for_exfil_tools(host)
        if tool_scan.get('tools_found'):
            for tool in tool_scan['tools_found']:
                removal_result = crowdstrike.remove_file(host, tool['path'])
                if removal_result.get('success'):
                    eradication_actions.append(f" Removed exfiltration tool: {tool['path']} from {host}")
                    print(f"   Removed exfiltration tool: {tool['path']} from {host}")
    except Exception as e:
        eradication_actions.append(f" Failed to scan/remove tools on {host}: {str(e)}")
        print(f"   Failed to scan/remove tools on {host}: {str(e)}")

# Clean up protocol abuse configurations
print("\n Cleaning up protocol abuse configurations...")
try:
    cleanup_config = {
        'reset_dns_settings': True,
        'disable_icmp_tunneling': True,
        'remove_protocol_handlers': True,
        'clean_registry_entries': True
    }
    cleanup_result = shuffle.cleanup_protocol_abuse(cleanup_config)
    if cleanup_result.get('success'):
        eradication_actions.append(" Protocol abuse configurations cleaned")
        print("   Protocol abuse configurations cleaned")
except Exception as e:
    eradication_actions.append(f" Failed to clean protocol configurations: {str(e)}")
    print(f"   Failed to clean protocol configurations: {str(e)}")

# Update firewall rules to prevent future exfiltration
print("\n Updating firewall rules...")
try:
    firewall_rules = {
        'block_icmp_exfiltration': True,
        'block_dns_tunneling': True,
        'block_large_unusual_transfers': True,
        'enable_protocol_inspection': True,
        'rate_limit_by_protocol': True
    }
    firewall_result = shuffle.update_firewall_rules(firewall_rules)
    if firewall_result.get('success'):
        eradication_actions.append(" Firewall rules updated")
        print("   Firewall rules updated")
except Exception as e:
    eradication_actions.append(f" Failed to update firewall rules: {str(e)}")
    print(f"   Failed to update firewall rules: {str(e)}")

# Implement content filtering
print("\n Implementing content filtering...")
try:
    filter_config = {
        'filter_dns_queries': True,
        'inspect_icmp_payloads': True,
        'block_encoded_data': True,
        'monitor_data_patterns': True
    }
    filter_result = shuffle.implement_content_filtering(filter_config)
    if filter_result.get('success'):
        eradication_actions.append(" Content filtering implemented")
        print("   Content filtering implemented")
except Exception as e:
    eradication_actions.append(f" Failed to implement content filtering: {str(e)}")
    print(f"   Failed to implement content filtering: {str(e)}")

# Verify exfiltration cessation
print("\n✅ Verifying exfiltration cessation...")
verification_results = []
current_exfil = splunk.execute_query(protocol_anomaly_query)
if len(current_exfil) == 0:
    verification_results.append(" Exfiltration traffic reduced to normal levels")
    print("   Exfiltration traffic reduced to normal levels")
else:
    verification_results.append(f" Exfiltration traffic still elevated: {len(current_exfil)} anomalies")
    print(f"   Exfiltration traffic still elevated: {len(current_exfil)} anomalies")

# Check for ongoing suspicious connections
ongoing_check = splunk.execute_query(icmp_exfil_query)
if len(ongoing_check) == 0:
    verification_results.append(" No ongoing ICMP exfiltration detected")
    print("   No ongoing ICMP exfiltration detected")
else:
    verification_results.append(f" Ongoing ICMP exfiltration: {len(ongoing_check)} instances")
    print(f"   Ongoing ICMP exfiltration: {len(ongoing_check)} instances")

# Update IRIS case with eradication actions
iris.update_case(case_id, {
    'eradication_actions': eradication_actions,
    'verification_results': verification_results,
    'eradication_timestamp': datetime.now().isoformat(),
    'status': 'eradication_complete'
})

print(f"\n Eradication Summary:")
print(f"  Actions Taken: {len([a for a in eradication_actions if a.startswith('')])}")
print(f"  Warnings: {len([a for a in eradication_actions if a.startswith('')])}")
print(f"  Errors: {len([a for a in eradication_actions if a.startswith('')])}")
print(f"  Verification Results: {len([r for r in verification_results if r.startswith('')])} passed")

# Share indicators with MISP
if len(exfil_indicators) > 0:
    misp.share_indicators(exfil_indicators, case_id)
    print(" Shared exfiltration indicators with MISP community")


## Phase 4: Recovery

### Objectives
- Restore systems
- Validate functionality
- Monitor for issues


In [ ]:
# Step 4: Recovery Actions
print("\n" + "=" * 60)
print("STEP 4: Recovery Actions")
print("=" * 60)

recovery_actions = []

# Re-enable isolated hosts
print("\n🔓 Re-enabling isolated hosts...")
for host in detection_summary['affected_hosts']:
    try:
        reenable_result = crowdstrike.reenable_host(host)
        if reenable_result.get('success'):
            recovery_actions.append(f" Re-enabled host: {host}")
            print(f"   Re-enabled host: {host}")
        else:
            recovery_actions.append(f" Failed to re-enable host: {host}")
            print(f"   Failed to re-enable host: {host}")
    except Exception as e:
        recovery_actions.append(f" Error re-enabling {host}: {str(e)}")
        print(f"   Error re-enabling {host}: {str(e)}")

# Restore normal network connectivity
print("\n🌊 Restoring normal network connectivity...")
try:
    restore_config = {
        'remove_traffic_throttling': True,
        'restore_normal_bandwidth': True,
        're_enable_blocked_protocols': False,  # Keep some restrictions
        'normal_operation_mode': True
    }
    restore_result = shuffle.restore_normal_connectivity(restore_config)
    if restore_result.get('success'):
        recovery_actions.append(" Normal network connectivity restored")
        print("   Normal network connectivity restored")
except Exception as e:
    recovery_actions.append(f" Failed to restore connectivity: {str(e)}")
    print(f"   Failed to restore connectivity: {str(e)}")

# Validate data integrity
print("\n Validating data integrity...")
integrity_checks = []
for host in detection_summary['affected_hosts']:
    try:
        integrity_result = shuffle.validate_data_integrity(host)
        if integrity_result.get('integrity_valid'):
            integrity_checks.append(f" Data integrity valid on {host}")
            print(f"   Data integrity valid on {host}")
        else:
            integrity_checks.append(f" Data integrity issues on {host}: {integrity_result.get('issues', [])}")
            print(f"   Data integrity issues on {host}: {integrity_result.get('issues', [])}")
    except Exception as e:
        integrity_checks.append(f" Integrity check failed on {host}: {str(e)}")
        print(f"   Integrity check failed on {host}: {str(e)}")

# Implement enhanced data protection
print("\n Implementing enhanced data protection...")
try:
    protection_config = {
        'enable_data_encryption': True,
        'implement_access_controls': True,
        'enable_audit_logging': True,
        'regular_backup_validation': True
    }
    protection_result = shuffle.implement_data_protection(protection_config)
    if protection_result.get('success'):
        recovery_actions.append(" Enhanced data protection implemented")
        print("   Enhanced data protection implemented")
except Exception as e:
    recovery_actions.append(f" Failed to implement data protection: {str(e)}")
    print(f"   Failed to implement data protection: {str(e)}")

# Monitor for exfiltration recurrence
print("\n Establishing monitoring for exfiltration recurrence...")
try:
    recurrence_monitoring = {
        'hosts': detection_summary['affected_hosts'],
        'duration_days': 30,
        'alert_threshold': 'medium',
        'indicators': ['protocol_anomalies', 'large_transfers', 'suspicious_connections']
    }
    splunk.setup_recurrence_monitoring(recurrence_monitoring)
    recovery_actions.append(" Recurrence monitoring established")
    print("   Recurrence monitoring established")
except Exception as e:
    recovery_actions.append(f" Failed to establish recurrence monitoring: {str(e)}")
    print(f"   Failed to establish recurrence monitoring: {str(e)}")

# Conduct data loss assessment
print("\n📊 Conducting data loss assessment...")
try:
    assessment_config = {
        'assess_exfiltrated_data': True,
        'identify_sensitive_data': True,
        'evaluate_business_impact': True,
        'generate_loss_report': True
    }
    assessment_result = shuffle.conduct_data_loss_assessment(assessment_config)
    if assessment_result.get('success'):
        recovery_actions.append(" Data loss assessment completed")
        print("   Data loss assessment completed")
        if assessment_result.get('data_lost'):
            print(f"   Potential data loss detected: {assessment_result.get('details', 'See assessment report')}")
    except Exception as e:
        recovery_actions.append(f" Data loss assessment failed: {str(e)}")
        print(f"   Data loss assessment failed: {str(e)}")

# Update IRIS case with recovery actions
iris.update_case(case_id, {
    'recovery_actions': recovery_actions,
    'integrity_checks': integrity_checks,
    'recovery_timestamp': datetime.now().isoformat(),
    'status': 'recovery_complete'
})

print(f"\n Recovery Summary:")
print(f"  Actions Taken: {len([a for a in recovery_actions if a.startswith('')])}")
print(f"  Warnings: {len([a for a in recovery_actions if a.startswith('')])}")
print(f"  Errors: {len([a for a in recovery_actions if a.startswith('')])}")
print(f"  Integrity Checks Passed: {len([c for c in integrity_checks if c.startswith('')])}")

# Trigger Shuffle workflow for recovery verification
shuffle.trigger_workflow('exfiltration_recovery_verification', {
    'case_id': case_id,
    'actions_taken': recovery_actions,
    'integrity_checks': integrity_checks
})
print(" Triggered recovery verification workflow")


## Phase 5: Post-Incident Activities

### Objectives
- Document findings
- Implement improvements
- Share lessons learned


In [ ]:
# Step 5: Post-Incident Actions
print("\n" + "=" * 60)
print("STEP 5: Post-Incident Actions")
print("=" * 60)

post_incident_actions = []

# Generate incident report
print("\n Generating incident report...")
try:
    incident_report = {
        'case_id': case_id,
        'title': 'Data Exfiltration Over Alternative Protocol Incident Response Report',
        'timeline': {
            'detection': detection_summary['timestamp'],
            'containment': datetime.now().isoformat(),  # Would be actual timestamp
            'eradication': datetime.now().isoformat(),   # Would be actual timestamp
            'recovery': datetime.now().isoformat(),      # Would be actual timestamp
            'closure': datetime.now().isoformat()
        },
        'impact_assessment': {
            'affected_hosts': len(detection_summary['affected_hosts']),
            'data_volume_exfiltrated': detection_summary['data_volume'],
            'protocols_abused': detection_summary['indicators']['protocol_anomalies'],
            'business_impact': 'HIGH',  # Would be determined by business impact analysis
            'data_sensitivity': 'HIGH',  # Would be determined by data classification
            'regulatory_impact': 'POTENTIAL'  # Would depend on data types exfiltrated
        },
        'response_metrics': {
            'detection_to_containment': 'TBD',  # Would calculate actual time
            'containment_to_eradication': 'TBD',
            'total_resolution_time': 'TBD',
            'automation_level': 'HIGH'
        },
        'attack_characteristics': {
            'attack_vector': 'Alternative Protocol Exfiltration',
            'protocols_used': ['ICMP', 'DNS', 'Custom Ports'],
            'data_volume': f"{detection_summary['data_volume']:,} bytes",
            'stealth_level': 'HIGH',
            'detection_difficulty': 'HIGH'
        },
        'lessons_learned': [
            'Alternative protocol exfiltration is hard to detect with traditional means',
            'Protocol-aware monitoring is essential for modern threats',
            'Traffic throttling effectively contained the exfiltration',
            'MISP integration provided valuable exfiltration indicators',
            'Content filtering prevented further data loss'
        ]
    }

    report_id = iris.generate_report(case_id, incident_report)
    post_incident_actions.append(f" Generated incident report: {report_id}")
    print(f"   Generated incident report: {report_id}")

except Exception as e:
    post_incident_actions.append(f" Failed to generate report: {str(e)}")
    print(f"   Failed to generate report: {str(e)}")

# Document lessons learned
print("\n Documenting lessons learned...")
lessons_learned = [
    "Implement protocol-aware network monitoring",
    "Regular DNS and ICMP traffic analysis",
    "Advanced anomaly detection for data transfers",
    "Content inspection for outbound traffic",
    "Regular security awareness training on exfiltration techniques",
    "Data classification and protection policies"
]

try:
    iris.add_lessons_learned(case_id, lessons_learned)
    post_incident_actions.append(" Documented lessons learned")
    print("   Documented lessons learned")
except Exception as e:
    post_incident_actions.append(f" Failed to document lessons learned: {str(e)}")
    print(f"   Failed to document lessons learned: {str(e)}")

# Implement preventive measures
print("\n Implementing preventive measures...")
preventive_measures = []

try:
    # Update exfiltration prevention policies
    policy_updates = {
        'protocol_inspection': 'deep',
        'data_loss_prevention': 'strict',
        'traffic_anomaly_detection': 'continuous',
        'content_filtering': 'enabled'
    }
    shuffle.update_security_policies(policy_updates)
    preventive_measures.append(" Updated exfiltration prevention policies")
    print("   Updated exfiltration prevention policies")

    # Enhance monitoring rules
    monitoring_updates = {
        'protocol_anomaly_detection': True,
        'dns_tunneling_monitoring': True,
        'icmp_traffic_analysis': True,
        'data_transfer_monitoring': True
    }
    splunk.update_monitoring_rules(monitoring_updates)
    preventive_measures.append(" Enhanced monitoring rules")
    print("   Enhanced monitoring rules")

    # Update threat intelligence feeds
    misp.update_feeds(['exfiltration', 'protocol-abuse', 'data-theft'])
    preventive_measures.append(" Updated threat intelligence feeds")
    print("   Updated threat intelligence feeds")

except Exception as e:
    preventive_measures.append(f" Failed to implement preventive measures: {str(e)}")
    print(f"   Failed to implement preventive measures: {str(e)}")

# Conduct post-incident review
print("\n Conducting post-incident review...")
try:
    review_findings = {
        'effectiveness_rating': 'HIGH',
        'areas_for_improvement': [
            'Earlier detection of protocol abuse',
            'Better data classification',
            'Enhanced content inspection capabilities'
        ],
        'recommendations': [
            'Implement AI-based protocol analysis',
            'Regular exfiltration simulation exercises',
            'Advanced network traffic analysis tools',
            'Zero-trust data access policies',
            'Automated data classification systems'
        ]
    }

    iris.conduct_post_incident_review(case_id, review_findings)
    post_incident_actions.append(" Conducted post-incident review")
    print("   Conducted post-incident review")

except Exception as e:
    post_incident_actions.append(f" Failed to conduct review: {str(e)}")
    print(f"   Failed to conduct review: {str(e)}")

# Close the incident case
print("\n Closing incident case...")
try:
    closure_data = {
        'closure_reason': 'Alternative protocol exfiltration contained - data loss prevented, monitoring enhanced',
        'closure_timestamp': datetime.now().isoformat(),
        'final_status': 'CLOSED',
        'follow_up_required': False,
        'reopen_criteria': 'Detection of similar alternative protocol exfiltration activity'
    }

    iris.close_case(case_id, closure_data)
    post_incident_actions.append(" Closed incident case")
    print("   Closed incident case")

except Exception as e:
    post_incident_actions.append(f" Failed to close case: {str(e)}")
    print(f"   Failed to close case: {str(e)}")

# Final summary
print(f"\n Post-Incident Summary:")
print(f"  Case ID: {case_id}")
print(f"  Actions Completed: {len([a for a in post_incident_actions if a.startswith('')])}")
print(f"  Warnings: {len([a for a in post_incident_actions if a.startswith('')])}")
print(f"  Errors: {len([a for a in post_incident_actions if a.startswith('')])}")
print(f"  Preventive Measures: {len(preventive_measures)}")
print(f"  Lessons Learned: {len(lessons_learned)}")

print("\n Incident Response Complete")
print("=" * 60)
print("Alternative protocol exfiltration incident successfully contained and resolved.")
print("Data loss prevented, monitoring enhanced, and preventive measures implemented.")
print("=" * 60)


## Summary

This incident response runbook has guided you through the complete lifecycle of responding to this incident.

### Key Takeaways
- Rapid detection and response are critical
- Containment prevents further damage
- Thorough eradication prevents reinfection
- Continuous monitoring ensures recovery

### Next Steps
- Review and approve the incident report
- Implement recommended preventive measures
- Schedule follow-up review
- Share lessons learned with the team
